In [1]:
import os
from typing import cast

from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import (
    AnalysisInput,
    AnalysisResult,
    AudioVisualContent,
    DocumentContent,
    AnalysisContent,
)
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from IPython.display import display, Markdown, Latex

load_dotenv()

endpoint = os.environ["AZURE_AI_ENDPOINT"]
key = os.getenv("AZURE_AI_API_KEY")
credential = AzureKeyCredential(key) if key else DefaultAzureCredential()

client = ContentUnderstandingClient(endpoint=endpoint, credential=credential)

## Analyze Audio

In [2]:
print("\n" + "=" * 60)
print("AUDIO ANALYSIS FROM URL")
print("=" * 60)
audio_url = "https://raw.githubusercontent.com/Azure-Samples/azure-ai-content-understanding-assets/main/audio/callCenterRecording.mp3"

print(f"Analyzing audio from URL with prebuilt-audioSearch...")
print(f"  URL: {audio_url}")

poller = client.begin_analyze(
    analyzer_id="prebuilt-audioSearch",
    inputs=[AnalysisInput(url=audio_url)],
)
result = poller.result()


AUDIO ANALYSIS FROM URL
Analyzing audio from URL with prebuilt-audioSearch...
  URL: https://raw.githubusercontent.com/Azure-Samples/azure-ai-content-understanding-assets/main/audio/callCenterRecording.mp3


In [3]:
# Cast AnalysisContent to AudioVisualContent to access audio/visual-specific properties
# AudioVisualContent derives from AnalysisContent and provides additional properties
# to access full information about audio/video, including timing, transcript phrases, and many others
audio_content = cast(AudioVisualContent, result.contents[0])
print("Markdown:")
display(Markdown(audio_content.markdown))

summary = audio_content.fields.get("Summary") if audio_content.fields else None
if summary and hasattr(summary, "value"):
    print(f"Summary: {summary.value}")

# Example: Access an additional field in AudioVisualContent (transcript phrases)
if audio_content.transcript_phrases and len(audio_content.transcript_phrases) > 0:
    print("Transcript (first two phrases):")
    for phrase in audio_content.transcript_phrases[:2]:
        print(f"  [{phrase.speaker}] {phrase.start_time_ms} ms: {phrase.text}")
# [END analyze_audio_from_url]

Markdown:


# Audio: 00:00.000 => 00:32.183

Transcript
```
WEBVTT

00:00.080 --> 00:00.640
<v Speaker 1>Good day.

00:00.880 --> 00:02.240
<v Speaker 1>Welcome to Contoso.

00:02.560 --> 00:03.760
<v Speaker 1>My name is John Doe.

00:03.920 --> 00:05.120
<v Speaker 1>How can I help you today?

00:05.440 --> 00:06.320
<v Speaker 2>Yes, good day.

00:06.640 --> 00:08.160
<v Speaker 2>My name is Maria Smith.

00:08.560 --> 00:11.360
<v Speaker 2>I would like to inquire about my current point balance.

00:11.680 --> 00:12.560
<v Speaker 1>No problem.

00:12.880 --> 00:13.920
<v Speaker 1>I am happy to help.

00:14.240 --> 00:16.720
<v Speaker 1>I need your date of birth to confirm your identity.

00:17.120 --> 00:19.600
<v Speaker 2>It is April 19th, 1988.

00:20.000 --> 00:20.480
<v Speaker 1>Great.

00:20.800 --> 00:24.160
<v Speaker 1>Your current point balance is 599 points.

00:24.560 --> 00:26.160
<v Speaker 1>Do you need any more information?

00:26.480 --> 00:27.200
<v Speaker 2>No, thank you.

00:27.600 --> 00:28.320
<v Speaker 2>That was all.

00:28.720 --> 00:29.360
<v Speaker 2>Goodbye.

00:29.680 --> 00:31.920
<v Speaker 1>You're welcome, goodbye a Cantoso.
```

Summary: The conversation is a customer service interaction where Maria Smith contacts Contoso to inquire about her current point balance. The agent, John Doe, verifies her identity by asking for her date of birth and then provides her with the information that she has 599 points. The customer confirms that she does not need any further information and ends the call politely.
Transcript (first two phrases):
  [Speaker 1] 80 ms: Good day.
  [Speaker 1] 880 ms: Welcome to Contoso.
